# Roihu – harjoitus 2: CNN-mallin parantaminen

Ensimmäisessä harjoituksessa ([cifar10_roihu.ipynb](cifar10_roihu.ipynb))
varmistettiin, että Roihun GPU-ympäristö toimii ja että yksinkertainen CNN
voidaan kouluttaa CIFAR-10-datalla. Tulokseksi saatiin noin **74 %** test accuracy.

Tässä harjoituksessa jatketaan samasta pohjasta ja yritetään parantaa mallia
hallitusti, vaihe vaiheelta.

---

## Mitä tässä harjoituksessa opitaan?

1. Datan **normalisointi**
2. **Data augmentation** eli koulutuskuvien satunnainen muokkaaminen
3. **Validointijoukko** koulutuksen seurantaan
4. Syvempi CNN, jossa on **BatchNorm** ja **Dropout**
5. **Learning rate scheduler**
6. Koulutuksen seuraaminen **oppimiskäyrillä**
7. Parhaan mallin **tallentaminen**
8. Tulosten tarkempi analyysi: **luokkakohtainen tarkkuus** ja **confusion matrix**
9. Mallin virheiden tarkastelu

---

## Ennen aloitusta

Käytä samoja Jupyter-asetuksia kuin ensimmäisessä harjoituksessa
(`gpuinteractive`, 4 CPU-ydintä, 16 GiB, `python-pytorch/2.13`).

Koulutus kestää tässä harjoituksessa pidempään kuin ensimmäisessä,
koska malli on suurempi ja epocheja on enemmän. Roihun GPU:lla
koko koulutus kestää tyypillisesti muutamia minuutteja.

## 1. Ympäristön tarkistus ja kirjastot

Tuodaan käyttöön samat kirjastot kuin ensimmäisessä harjoituksessa sekä muutama uusi:

- `time` – koulutuksen keston mittaamiseen
- `Subset` – datasetin jakamiseen koulutus- ja validointiosaan

Asetamme myös **satunnaislukusiemenen** (`seed`). Sen ansiosta
datan jako ja mallin alkupainot ovat samat joka ajokerralla, jolloin
eri kokeiluja on helpompi verrata keskenään.

In [ ]:
import time
import random

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset

SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch version:", torch.__version__)
print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU ei ole käytettävissä.")

## 2. Datan normalisointi

Ensimmäisessä harjoituksessa käytettiin pelkkää `ToTensor()`-muunnosta,
joka skaalaa pikseliarvot välille 0–1.

Neuroverkko oppii yleensä nopeammin ja vakaammin, kun syötteen jokaisella
värikanavalla on:

- keskiarvo noin **0**
- keskihajonta noin **1**

Tätä kutsutaan **normalisoinniksi**:

`normalisoitu = (pikseli - keskiarvo) / keskihajonta`

Lasketaan keskiarvo ja keskihajonta jokaiselle värikanavalle (R, G, B)
**pelkästä koulutusdatasta**. Testidataa ei käytetä tähän, koska
testidatan pitää pysyä mallille täysin näkymättömänä.

In [ ]:
raw_trainset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True
)

# raw_trainset.data on NumPy-taulukko muotoa (50000, 32, 32, 3)
pixels = raw_trainset.data / 255.0

MEAN = tuple(pixels.mean(axis=(0, 1, 2)).tolist())
STD = tuple(pixels.std(axis=(0, 1, 2)).tolist())

print("Data shape:", raw_trainset.data.shape)
print("Mean (R, G, B):", [round(m, 4) for m in MEAN])
print("Std  (R, G, B):", [round(s, 4) for s in STD])

## 3. Data augmentation

Malli näkee koulutuksen aikana samat 50 000 kuvaa uudelleen jokaisella
epochilla. Tällöin on vaarana, että malli alkaa **muistaa** koulutuskuvat
sen sijaan, että se oppisi yleisiä piirteitä. Tätä kutsutaan
**ylisovittamiseksi** (overfitting).

**Data augmentation** muokkaa koulutuskuvia satunnaisesti joka kerta,
kun kuva syötetään mallille. Käytämme kahta yksinkertaista muunnosta:

- `RandomCrop(32, padding=4)` – kuvaa siirretään satunnaisesti muutaman pikselin verran
- `RandomHorizontalFlip()` – kuva peilataan vaakasuunnassa 50 %:n todennäköisyydellä

Kissa on edelleen kissa, vaikka kuva olisi peilattu tai hieman siirretty.
Näin malli oppii, että luokka ei riipu kohteen tarkasta sijainnista.

> **Tärkeää:** augmentaatiota käytetään **vain koulutusdatassa**.
> Validointi- ja testidata pidetään muuttumattomina, jotta tulokset ovat vertailukelpoisia.

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

Katsotaan, miltä augmentaatio näyttää käytännössä.

Ensimmäinen kuva on alkuperäinen, ja muut ovat saman kuvan satunnaisia
augmentoituja versioita. Normalisointi jätetään tässä pois, jotta
kuvat näyttävät luonnollisilta.

In [ ]:
augment_only = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

classes = raw_trainset.classes
index = random.randint(0, len(raw_trainset) - 1)
pil_image, label = raw_trainset[index]

plt.figure(figsize=(15, 3))

plt.subplot(1, 6, 1)
plt.imshow(pil_image)
plt.title(f"Alkuperäinen\n({classes[label]})")
plt.axis("off")

for i in range(5):
    augmented = augment_only(pil_image)

    plt.subplot(1, 6, i + 2)
    plt.imshow(augmented.permute(1, 2, 0))
    plt.title(f"Augmentoitu {i + 1}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## 4. Koulutus-, validointi- ja testidata

Ensimmäisessä harjoituksessa malli arvioitiin suoraan testidatalla.

Jos testidataa käytetään toistuvasti mallin säätämiseen (esim. epochien
määrän tai learning raten valintaan), mallia alkaa huomaamatta optimoida
juuri testidataa varten. Silloin test accuracy ei enää kerro rehellisesti,
miten malli toimii uudella datalla.

Siksi jaamme datan kolmeen osaan:

| Osa | Koko | Käyttötarkoitus |
| --- | --- | --- |
| **Train** | 45 000 | Mallin painojen kouluttaminen |
| **Validation** | 5 000 | Koulutuksen seuraaminen ja parhaan mallin valinta |
| **Test** | 10 000 | Lopullinen arvio, käytetään vasta lopussa |

Koulutus- ja validointiosa otetaan alkuperäisestä 50 000 kuvan
koulutusdatasta. Koska validointidataan ei saa kohdistaa augmentaatiota,
luomme koulutusdatasta kaksi versiota eri muunnoksilla ja valitsemme
niistä eri indeksit.

In [ ]:
full_train_augmented = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=train_transform
)

full_train_plain = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=test_transform
)

testset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=test_transform
)

VAL_SIZE = 5000

generator = torch.Generator().manual_seed(SEED)
indices = torch.randperm(len(full_train_plain), generator=generator).tolist()

trainset = Subset(full_train_augmented, indices[VAL_SIZE:])
valset = Subset(full_train_plain, indices[:VAL_SIZE])

print("Training images:  ", len(trainset))
print("Validation images:", len(valset))
print("Test images:      ", len(testset))

### DataLoaderit

DataLoaderit luodaan samalla tavalla kuin ensimmäisessä harjoituksessa.

Vain koulutusdata sekoitetaan (`shuffle=True`). Validointi- ja testidatan
järjestyksellä ei ole merkitystä.

`num_workers=4` vastaa Jupyter-sessioon varattua neljää CPU-ydintä.
Augmentaatio tehdään CPU:lla, joten näistä prosesseista on nyt erityisesti hyötyä.

In [ ]:
BATCH_SIZE = 128

trainloader = DataLoader(
    trainset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

valloader = DataLoader(
    valset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

testloader = DataLoader(
    testset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

images, labels = next(iter(trainloader))

print("Image batch:", images.shape)
print("Label batch:", labels.shape)
print(f"Pixel mean after normalization: {images.mean():.3f}")
print(f"Pixel std after normalization:  {images.std():.3f}")

Normalisoinnin jälkeen pikseleiden keskiarvon pitäisi olla lähellä **0**
ja keskihajonnan lähellä **1**.

## 5. Parannettu CNN

Ensimmäisen harjoituksen mallissa oli kolme konvoluutiokerrosta.
Nyt rakennetaan hieman syvempi malli ja lisätään kaksi uutta kerrostyyppiä.

### BatchNorm

`BatchNorm2d` normalisoi kerroksen ulostulot jokaisen batchin sisällä.
Tämä tasapainottaa koulutusta ja sallii yleensä suuremman learning raten,
jolloin malli oppii nopeammin.

### Dropout

`Dropout` nollaa koulutuksen aikana satunnaisesti osan neuroneista.
Malli ei voi silloin nojata liikaa yksittäisiin neuroneihin, mikä
vähentää ylisovittamista.

Dropout on käytössä vain koulutuksen aikana. Arvioinnissa
`model.eval()` kytkee sen pois päältä.

### Rakenne

Malli koostuu kolmesta lohkosta. Jokaisessa lohkossa on kaksi
konvoluutiokerrosta, jonka jälkeen kuvan kokoa pienennetään:

| Lohko | Kanavat | Kuvan koko |
| --- | --- | --- |
| Syöte | 3 | 32 × 32 |
| Lohko 1 | 64 | 16 × 16 |
| Lohko 2 | 128 | 8 × 8 |
| Lohko 3 | 256 | 4 × 4 |

Toistuva rakenne kirjoitetaan apufunktioksi `conv_block`, jotta koodia ei
tarvitse toistaa.

In [ ]:
def conv_block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True)
    )


class BetterCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            conv_block(3, 64),
            conv_block(64, 64),
            nn.MaxPool2d(2),
            nn.Dropout(0.1),

            conv_block(64, 128),
            conv_block(128, 128),
            nn.MaxPool2d(2),
            nn.Dropout(0.2),

            conv_block(128, 256),
            conv_block(256, 256),
            nn.MaxPool2d(2),
            nn.Dropout(0.3)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = BetterCNN().to(device)

num_parameters = sum(p.numel() for p in model.parameters())

print("Device:", device)
print(f"Parameters: {num_parameters:,}")

Ensimmäisen harjoituksen mallissa oli noin **620 000** parametria.
Uudessa mallissa niitä on noin **2,2 miljoonaa** eli yli kolme kertaa enemmän.

Isompi malli pystyy oppimaan monimutkaisempia piirteitä, mutta se myös
ylisovittaa helpommin. Siksi augmentaatio ja Dropout ovat tässä tärkeitä.

## 6. Optimointi ja learning rate scheduler

Ensimmäisessä harjoituksessa learning rate oli koko koulutuksen ajan sama (`0.001`).

Usein on parempi muuttaa learning ratea koulutuksen aikana:

- **alussa** learning ratea nostetaan vähitellen, jotta koulutus käynnistyy vakaasti
- **keskellä** suuri learning rate auttaa löytämään hyvän ratkaisun nopeasti
- **lopussa** pieni learning rate hienosäätää painoja

Käytämme tähän `OneCycleLR`-scheduleria, joka päivittää learning ratea
jokaisen batchin jälkeen.

Optimointialgoritmina on `AdamW`, joka on Adamin muunnelma. Sen
`weight_decay`-parametri pitää mallin painot pieninä, mikä sekin
vähentää ylisovittamista.

In [ ]:
EPOCHS = 20

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=5e-4
)

scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=0.003,
    epochs=EPOCHS,
    steps_per_epoch=len(trainloader)
)

print("Epochs:", EPOCHS)
print("Batches per epoch:", len(trainloader))
print("Total training steps:", EPOCHS * len(trainloader))

## 7. Koulutus- ja arviointifunktiot

Ensimmäisessä harjoituksessa koulutus ja testaus kirjoitettiin suoraan
silmukoiksi. Nyt ne kirjoitetaan **funktioiksi**, koska niitä käytetään
useaan kertaan:

- `train_one_epoch` kouluttaa mallia yhden epochin ajan
- `evaluate` laskee mallin lossin ja tarkkuuden annetulla datalla

Molemmat palauttavat **keskimääräisen lossin** ja **accuracyn**.

In [ ]:
def train_one_epoch(model, loader):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()
        scheduler.step()

        running_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, 100 * correct / total


def evaluate(model, loader):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, 100 * correct / total

## 8. Mallin kouluttaminen

Jokaisen epochin jälkeen:

1. malli arvioidaan validointidatalla
2. tulokset tallennetaan `history`-sanakirjaan oppimiskäyriä varten
3. jos validointitarkkuus on paras tähän mennessä, mallin painot
   tallennetaan tiedostoon `best_model.pt`

Paras epoch merkitään tulosteessa tähdellä `*`.

Mittaamme myös jokaisen epochin keston, jotta näemme, kuinka nopeasti
Roihun GPU kouluttaa mallia.

In [ ]:
history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "lr": []
}

best_val_acc = 0.0
training_start = time.time()

for epoch in range(EPOCHS):
    epoch_start = time.time()

    current_lr = optimizer.param_groups[0]["lr"]

    train_loss, train_acc = train_one_epoch(model, trainloader)
    val_loss, val_acc = evaluate(model, valloader)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["lr"].append(current_lr)

    marker = ""
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pt")
        marker = " *"

    epoch_time = time.time() - epoch_start

    print(
        f"Epoch {epoch + 1:2d}/{EPOCHS} | "
        f"Train loss: {train_loss:.4f}, acc: {train_acc:5.2f}% | "
        f"Val loss: {val_loss:.4f}, acc: {val_acc:5.2f}% | "
        f"{epoch_time:4.1f} s{marker}"
    )

training_time = time.time() - training_start

print()
print(f"Training time: {training_time / 60:.1f} min")
print(f"Best validation accuracy: {best_val_acc:.2f}%")

if torch.cuda.is_available():
    max_memory = torch.cuda.max_memory_allocated() / 1024**3
    print(f"Max GPU memory used: {max_memory:.2f} GiB")

## 9. Oppimiskäyrät

Oppimiskäyristä näkee paljon enemmän kuin pelkästä lopputuloksesta.

**Miten käyriä luetaan?**

- Jos **train loss** laskee mutta **validation loss** alkaa nousta,
  malli **ylisovittaa**: se oppii koulutusdatan ulkoa.
- Jos molemmat lossit pysyvät korkeina, malli **alisovittaa**:
  se on liian yksinkertainen tai koulutus on liian lyhyt.
- Jos molemmat laskevat ja pysyvät lähellä toisiaan, koulutus sujuu hyvin.

Huomaa, että tässä **train accuracy voi olla jopa pienempi kuin validation accuracy**.
Tämä johtuu siitä, että koulutuskuvat ovat augmentoituja (vaikeampia) ja
Dropout on päällä vain koulutuksen aikana.

In [ ]:
epochs_range = range(1, EPOCHS + 1)

plt.figure(figsize=(16, 4))

plt.subplot(1, 3, 1)
plt.plot(epochs_range, history["train_loss"], label="Train")
plt.plot(epochs_range, history["val_loss"], label="Validation")
plt.title("Loss")
plt.xlabel("Epoch")
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 3, 2)
plt.plot(epochs_range, history["train_acc"], label="Train")
plt.plot(epochs_range, history["val_acc"], label="Validation")
plt.title("Accuracy (%)")
plt.xlabel("Epoch")
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 3, 3)
plt.plot(epochs_range, history["lr"])
plt.title("Learning rate (epochin alussa)")
plt.xlabel("Epoch")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Lopullinen testaus

Nyt ladataan **paras** tallennettu malli ja arvioidaan se testidatalla.

Testidataa käytetään tässä harjoituksessa **vain kerran**, aivan lopussa.
Näin saadaan rehellinen arvio siitä, miten malli toimii täysin uudella datalla.

Vertaa tulosta ensimmäisen harjoituksen noin 74 %:iin.

In [ ]:
model.load_state_dict(torch.load("best_model.pt", map_location=device))

test_loss, test_acc = evaluate(model, testloader)

print(f"Test loss:     {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.2f}%")

## 11. Luokkakohtainen tarkkuus ja confusion matrix

Yksi accuracy-luku ei kerro, **missä** malli onnistuu ja missä se tekee virheitä.

**Confusion matrix** on taulukko, jossa:

- **rivi** kertoo kuvan oikean luokan
- **sarake** kertoo mallin ennustaman luokan
- **diagonaali** sisältää oikeat ennusteet

Diagonaalin ulkopuoliset suuret luvut paljastavat, mitkä luokat
sekoittuvat mallilla keskenään.

Kerätään ensin kaikki testidatan ennusteet.

In [ ]:
model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in testloader:
        images = images.to(device, non_blocking=True)

        outputs = model(images)

        all_predictions.append(outputs.argmax(1).cpu())
        all_labels.append(labels)

all_predictions = torch.cat(all_predictions)
all_labels = torch.cat(all_labels)

num_classes = len(classes)

confusion = torch.bincount(
    all_labels * num_classes + all_predictions,
    minlength=num_classes ** 2
).reshape(num_classes, num_classes)

class_accuracy = 100 * confusion.diag() / confusion.sum(dim=1)

for name, acc in zip(classes, class_accuracy):
    print(f"{name:>10}: {acc:5.1f}%")

In [ ]:
plt.figure(figsize=(9, 8))
plt.imshow(confusion, cmap="Blues")
plt.colorbar()

plt.xticks(range(num_classes), classes, rotation=45, ha="right")
plt.yticks(range(num_classes), classes)
plt.xlabel("Ennustettu luokka")
plt.ylabel("Oikea luokka")
plt.title("Confusion matrix (testidata)")

threshold = confusion.max() / 2

for row in range(num_classes):
    for col in range(num_classes):
        value = confusion[row, col].item()
        plt.text(
            col, row, value,
            ha="center", va="center",
            color="white" if value > threshold else "black",
            fontsize=8
        )

plt.tight_layout()
plt.show()

### Mitä confusion matrixista kannattaa etsiä?

Tyypillisesti CIFAR-10:ssä vaikeimpia luokkia ovat **cat** ja **dog**,
jotka sekoittuvat keskenään. Myös **automobile** ja **truck** sekä
**bird**, **deer** ja **frog** voivat mennä sekaisin.

Tämä on ymmärrettävää: 32 × 32 pikselin kuvassa kissan ja koiran
erottaminen on vaikeaa ihmisellekin.

## 12. Mallin virheiden tarkastelu

Lopuksi katsotaan muutamia kuvia, jotka malli luokitteli väärin.

Kuvat on normalisoitu, joten ne pitää ennen näyttämistä
**palauttaa** alkuperäiselle asteikolle:

`kuva = normalisoitu * keskihajonta + keskiarvo`

In [ ]:
mean_tensor = torch.tensor(MEAN).view(3, 1, 1)
std_tensor = torch.tensor(STD).view(3, 1, 1)

wrong_indices = (all_predictions != all_labels).nonzero().flatten().tolist()
print("Väärin luokiteltuja testikuvia:", len(wrong_indices))

sample = random.sample(wrong_indices, 10)

plt.figure(figsize=(15, 6))

for i, index in enumerate(sample):
    image, label = testset[index]
    image = (image * std_tensor + mean_tensor).clamp(0, 1)

    predicted = all_predictions[index].item()

    plt.subplot(2, 5, i + 1)
    plt.imshow(image.permute(1, 2, 0))
    plt.title(f"Oikea: {classes[label]}\nEnnuste: {classes[predicted]}", fontsize=10)
    plt.axis("off")

plt.tight_layout()
plt.show()

Osa virheistä on todennäköisesti sellaisia, että ihmisenkin olisi
vaikea tunnistaa kuvaa oikein. Tämä kertoo myös jotain datan
rajoituksista: 32 × 32 pikseliä on hyvin pieni kuva.

## 13. Yhteenveto

Ensimmäiseen harjoitukseen verrattuna teimme seuraavat muutokset:

| Muutos | Mitä se tekee? |
| --- | --- |
| Normalisointi | Tasapainottaa syötteen arvot ja nopeuttaa oppimista |
| Data augmentation | Monipuolistaa koulutusdataa ja vähentää ylisovittamista |
| Validointijoukko | Mahdollistaa koulutuksen seuraamisen ilman testidatan käyttöä |
| BatchNorm | Vakauttaa ja nopeuttaa koulutusta |
| Dropout | Vähentää ylisovittamista |
| Syvempi malli | Oppii monimutkaisempia piirteitä |
| OneCycleLR | Muuttaa learning ratea koulutuksen aikana |
| Parhaan mallin tallennus | Valitsee mallin validointitarkkuuden perusteella |

Lisäksi opimme analysoimaan tulosta tarkemmin oppimiskäyrien,
luokkakohtaisen tarkkuuden ja confusion matrixin avulla.

Työketju näyttää nyt tältä:

**Data**
↓
**Normalisointi + augmentaatio**
↓
**Train / Validation / Test**
↓
**Parannettu CNN**
↓
**Koulutus + validointi jokaisen epochin jälkeen**
↓
**Paras malli talteen**
↓
**Testaus ja analyysi**

## 14. Kokeile seuraavaksi

Muuta yhtä asiaa kerrallaan ja vertaa validointitarkkuutta.
Näin näet, mikä vaikutus kullakin muutoksella on.

### 1. Poista augmentaatio

Poista `RandomCrop` ja `RandomHorizontalFlip` `train_transform`-muunnoksesta.

Miten oppimiskäyrät muuttuvat? Alkaako validation loss nousta?

### 2. Muuta epochien määrää

Kokeile `EPOCHS = 10` ja `EPOCHS = 40`.

Kuinka paljon pidempi koulutus parantaa tulosta? Entä kuinka paljon se vie aikaa?

### 3. Muuta learning ratea

Kokeile esimerkiksi `max_lr=0.001` tai `max_lr=0.01`.

### 4. Kokeile mixed precision -koulutusta

Nykyaikaiset GPU:t, kuten Roihun NVIDIA GH200, laskevat nopeammin
pienemmällä tarkkuudella (`bfloat16`). Muuta `train_one_epoch`-funktiossa
mallin kutsu muotoon:

```python
with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    outputs = model(images)
    loss = criterion(outputs, labels)
```

Muuttuuko epochin kesto? Entä tarkkuus?

### 5. Kasvata batch sizea

GH200-GPU:lla on paljon muistia. Kokeile `BATCH_SIZE = 512`.

Huomaa, että suurempi batch size vaatii usein myös suuremman learning raten.

## 15. Lopuksi

Muista sulkea Jupyter-sessio, kun lopetat työskentelyn, jotta varatut
laskentaresurssit vapautuvat:

**My Interactive Sessions → Oma Jupyter session → Cancel**

Tallennettu malli `best_model.pt` jää Roihuun työhakemistoosi.
Sen voi ladata myöhemmin uudelleen komennolla:

```python
model = BetterCNN().to(device)
model.load_state_dict(torch.load("best_model.pt", map_location=device))
```